# HMM Training mit CmdStanPy (Parallel)

Dieses Notebook trainiert **nur HMM** Modelle mit S=2,3,4.

**Vorteile von CmdStanPy:**

- ✓ Chains laufen parallel (statt sequenziell)
- ✓ ~2x schneller bei 2 Chains
- ✓ Bessere Performance

**Wichtig:** Lassen Sie parallel das VD-HMM-Notebook laufen für maximale Effizienz!


In [1]:
import sys
from pathlib import Path

# Add project root to path
sys.path.append(str(Path("..").resolve()))

import pickle
import numpy as np
from helpers import ModelData
import cmdstanpy

print(f"CmdStanPy Version: {cmdstanpy.__version__}")
print(f"CmdStan Path: {cmdstanpy.cmdstan_path()}")

CmdStanPy Version: 1.3.0
CmdStan Path: /Users/omidsedighi-mornani/.cmdstan/cmdstan-2.37.0


/Users/omidsedighi-mornani/Desktop/STUDIUM WIRTSCHAFTSINFORMATIK/Semester 5/Seminar Gust/SourceCode/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Load processed data
data_path = "../data/processed/processed_data.pkl"
model_data = ModelData.from_pickle(data_path)

print(model_data.summary())


ModelData Summary:
States (HMM): Not set
Total businesses: 921
Training samples: 500
Total observations: 64887
Number of covariates: 16

Data shapes:
- Ratings: 64887
- Sentiment: 64887
- Days: 64887
- Q matrix: (500, 16)
- R matrix: (16, 16)
- X_test: (421, 16)

Business status:
- Closed: 225
- Open: 696

Preprocessing artifacts: Available
- Train indices: 500
- Calibration indices: 200
- Eval indices: 221

Benchmark data: Available
- Benchmark covariates shape: (921, 23)
- Columns: business_id, density, Checkin, category, chain...



In [3]:
# Configuration
stan_model_folder = Path("../data/stan_code")
fitted_model_folder = Path("../models")
fitted_model_folder.mkdir(parents=True, exist_ok=True)

print(f"✓ Configuration set")
print(f"  Stan models: {stan_model_folder}")
print(f"  Output folder: {fitted_model_folder}")

✓ Configuration set
  Stan models: ../data/stan_code
  Output folder: ../models


In [4]:
from helpers import prepare_stan_data


def train_model_cmdstan(
    model_data,
    S,
    model_name="hmm",
    chains=2,
    parallel_chains=2,
    iter_warmup=500,
    iter_sampling=500,
    seed=42,
    adapt_delta=0.8,
    max_treedepth=10,
):
    """
    Trainiert ein Modell mit CmdStanPy.

    Parameters:
    -----------
    model_data : ModelData
        Daten für das Training
    S : int
        Anzahl der Hidden States (2-5)
    model_name : str
        'vdhmm' oder 'hmm'
    chains : int
        Anzahl der MCMC Chains
    parallel_chains : int
        Anzahl parallel laufender Chains (nutzt parallel_chains CPU Cores)
    iter_warmup : int
        Warmup Iterationen
    iter_sampling : int
        Sampling Iterationen (post-warmup)
    seed : int
        Random Seed
    adapt_delta : float
        Stan adapt_delta Parameter (0.8-0.99, höher = konservativer)
    max_treedepth : int
        Stan max_treedepth Parameter

    Returns:
    --------
    cmdstanpy.CmdStanMCMC : Fit-Objekt
    """
    assert model_name in ["vdhmm", "hmm"], f"Invalid model_name: {model_name}"
    assert S in range(2, 6), "S must be between 2 and 5"

    # Daten vorbereiten
    stan_data = prepare_stan_data(model_data, S)

    # Model file
    model_file = stan_model_folder / f"{model_name}.stan"
    if not model_file.exists():
        raise FileNotFoundError(f"Stan model not found: {model_file}")

    print(f"\n{'='*60}")
    print(f"Training {model_name.upper()} with S={S} states (CmdStanPy)")
    print(f"{'='*60}")
    print(f"Model file: {model_file}")
    print(f"\nConfiguration:")
    print(f"  Chains: {chains}")
    print(f"  Parallel chains: {parallel_chains}")
    print(f"  Warmup iterations: {iter_warmup}")
    print(f"  Sampling iterations: {iter_sampling}")
    print(f"  Total iterations: {iter_warmup + iter_sampling}")
    print(f"  Seed: {seed}")
    print(f"  Adapt delta: {adapt_delta}")
    print(f"  Max treedepth: {max_treedepth}")

    # Compile model
    print(f"\nCompiling model...")
    model = cmdstanpy.CmdStanModel(stan_file=str(model_file))
    print(f"✓ Model compiled")

    # Sample
    print(f"\nSampling...")
    fit = model.sample(
        data=stan_data,
        chains=chains,
        parallel_chains=parallel_chains,
        iter_warmup=iter_warmup,
        iter_sampling=iter_sampling,
        seed=seed,
        adapt_delta=adapt_delta,
        max_treedepth=max_treedepth,
        show_progress=True,
    )

    print(f"\n✓ Sampling complete!")

    # Save model
    output_path = fitted_model_folder / f"{model_name}_{S}_cmdstan.pkl"
    with open(output_path, "wb") as f:
        pickle.dump(
            {
                "fit": fit,
                "model_name": model_name,
                "S": S,
                "stan_data": stan_data,
                "summary": fit.summary(),
            },
            f,
        )

    print(f"✓ Model saved to {output_path}")

    # Diagnostics
    print(f"\n{'-'*60}")
    print("Diagnostics:")
    print(f"{'-'*60}")
    print(fit.diagnose())

    # Summary statistics
    print(f"\n{'-'*60}")
    print("Summary (first 20 parameters):")
    print(f"{'-'*60}")
    summary_df = fit.summary()
    print(summary_df.head(20))

    return fit


print("✓ Training function defined (CmdStanPy)")

✓ Training function defined (CmdStanPy)


## Training Configuration

**Paper-Standard:** 2 chains × 1000 iterations (500 warmup + 500 sampling)

**Für schnelles Testen:** 2 chains × 200 iterations (100 warmup + 100 sampling)


In [5]:
# Training Settings
SEED = 42
CHAINS = 4
PARALLEL_CHAINS = 4  # Nutzt 4 CPU Cores parallel
ITER_WARMUP = 1000  # Paper: 500, Quick test: 100
ITER_SAMPLING = 1000  # Paper: 500, Quick test: 100
ADAPT_DELTA = 0.95  # 0.8-0.95, höher falls divergent transitions
MAX_TREEDEPTH = 10  # 10-15

np.random.seed(SEED)

print("Training Configuration:")
print(f"  Seed: {SEED}")
print(f"  Chains: {CHAINS}")
print(f"  Parallel chains: {PARALLEL_CHAINS}")
print(f"  Warmup iterations: {ITER_WARMUP}")
print(f"  Sampling iterations: {ITER_SAMPLING}")
print(f"  Total iterations: {ITER_WARMUP + ITER_SAMPLING}")
print(f"  Total posterior samples: {CHAINS * ITER_SAMPLING}")

Training Configuration:
  Seed: 42
  Chains: 4
  Parallel chains: 4
  Warmup iterations: 1000
  Sampling iterations: 1000
  Total iterations: 2000
  Total posterior samples: 4000


## Training HMM Models (S=2, 3, 4)

Standard Hidden Markov Models mit zeitunabhängigen Übergangswahrscheinlichkeiten.


In [6]:
# Dictionary zum Speichern aller Modelle
trained_models = {}

# HMM Training für S=2 bis S=4
for S in range(2, 4 + 1):
    try:
        print(f"\n\n{'#'*60}")
        print(f"# HMM Training: S={S}")
        print(f"{'#'*60}\n")

        fit = train_model_cmdstan(
            model_data=model_data,
            S=S,
            model_name="hmm",
            chains=CHAINS,
            parallel_chains=PARALLEL_CHAINS,
            iter_warmup=ITER_WARMUP,
            iter_sampling=ITER_SAMPLING,
            seed=SEED,
            adapt_delta=ADAPT_DELTA,
            max_treedepth=MAX_TREEDEPTH,
        )

        trained_models[f"hmm_{S}"] = fit
        print(f"\n✓✓✓ HMM with S={S} completed successfully! ✓✓✓\n")

    except Exception as e:
        print(f"\n✗✗✗ Error training HMM with S={S}: {e} ✗✗✗\n")
        raise

print("\n" + "=" * 60)
print("HMM Training Complete!")
print("=" * 60)



############################################################
# HMM Training: S=2
############################################################


Training HMM with S=2 states (CmdStanPy)
Model file: ../data/stan_code/hmm.stan

Configuration:
  Chains: 4
  Parallel chains: 4
  Warmup iterations: 1000
  Sampling iterations: 1000
  Total iterations: 2000
  Seed: 42
  Adapt delta: 0.95
  Max treedepth: 10

Compiling model...
✓ Model compiled

Sampling...


17:06:48 - cmdstanpy - INFO - CmdStan start processing
chain 1:   0%|          | 0/2000 [00:00<?, ?it/s, (Warmup)]


chain 1:   0%|          | 1/2000 [00:00<18:30,  1.80it/s, (Warmup)]


chain 1:   5%|▌         | 100/2000 [08:09<2:35:32,  4.91s/it, (Warmup)]

chain 1:  10%|█         | 200/2000 [12:45<1:49:12,  3.64s/it, (Warmup)]


chain 1:  15%|█▌        | 300/2000 [17:01<1:29:13,  3.15s/it, (Warmup)]


chain 1:  20%|██        | 400/2000 [21:26<1:18:44,  2.95s/it, (Warmup)]


chain 1:  25%|██▌       | 500/2000 [26:03<1:12:08,  2.89s/it, (Warmup)]

chain 1:  30%|███       | 600/2000 [30:31<1:05:42,  2.82s/it, (Warmup)]



chain 1:  35%|███▌      | 700/2000 [37:38<1:11:17,  3.29s/it, (Warmup)]

chain 1:  40%|████      | 800/2000 [40:49<57:01,  2.85s/it, (Warmup)]  


chain 1:  45%|████▌     | 900/2000 [44:48<49:38,  2.71s/it, (Warmup)]


chain 1:  55%|█████▌    | 1100/2000 [51:59<35:05,  2.34s/it, (Sampling)]





chain 1:  60%|██████    | 1200/2000 [55:10<29:09,  2.19s/it, (Sampling)]



18:54:02 - cmdstanpy - INFO - CmdStan done processing.
18:54:02 - cmdstanpy - WARNING - Non-fatal error during sampling:
Exception: ordered_probit: Location parameter is inf, but must be finite! (in 'hmm.stan', line 172, column 8 to column 74)
	Exception: ordered_probit: Location parameter is inf, but must be finite! (in 'hmm.stan', line 172, column 8 to column 74)
	Exception: ordered_probit: Location parameter is inf, but must be finite! (in 'hmm.stan', line 172, column 8 to column 74)
	Exception: ordered_probit: Cut-points is not a valid ordered vector. The element at 2 is -29.08, but should be greater than the previous element, -29.08 (in 'hmm.stan', line 172, column 8 to column 74)
	Exception: ordered_probit: Cut-points is not a valid ordered vector. The element at 2 is -14.3187, but should be greater than the previous element, -14.3187 (in 'hmm.stan', line 172, column 8 to column 74)
	Exception: bernoulli_logit_lpmf: Logit transformed probability parameter[1] is nan, but must be 

18:54:03 - cmdstanpy - WARNING - Some chains may have failed to converge.
	Chain 1 had 4 divergent transitions (0.4%)
	Chain 2 had 2 divergent transitions (0.2%)
	Chain 3 had 1 divergent transitions (0.1%)
	Chain 4 had 1 divergent transitions (0.1%)
	Use the "diagnose()" method on the CmdStanMCMC object to see further information.



✓ Sampling complete!
✓ Model saved to ../models/hmm_2_cmdstan.pkl

------------------------------------------------------------
Diagnostics:
------------------------------------------------------------
Checking sampler transitions treedepth.
Treedepth satisfactory for all transitions.

Checking sampler transitions for divergences.
8 of 4000 (0.20%) transitions ended with a divergence.
These divergent transitions indicate that HMC is not fully able to explore the posterior distribution.
Try increasing adapt delta closer to 1.
If this doesn't remove all divergences, try to reparameterize the model.

Checking E-BFMI - sampler transitions HMC potential energy.
E-BFMI satisfactory.

Rank-normalized split effective sample size satisfactory for all parameters.

Rank-normalized split R-hat values satisfactory for all parameters.

Processing complete.


------------------------------------------------------------
Summary (first 20 parameters):
--------------------------------------------------

...

                               Mean       MCSE      StdDev       MAD  \
lp__                  -46047.200000  72.529900  102.810000  7.348510   
pi[1]                      0.164050   0.003918    0.023617  0.023791   
pi[2]                      0.466348   0.040196    0.063507  0.048574   
pi[3]                      0.369602   0.044222    0.068200  0.043598   
tpm[1,1]                   0.714514   0.254112    0.364260  0.099823   
tpm[1,2]                   0.285486   0.254112    0.364260  0.099823   
tpm[2,1]                   0.459560   0.133827    0.200856  0.111228   
tpm[2,2]                   0.540440   0.133827    0.200856  0.111228   
tpm[3,1]                   0.189014   0.204199    0.290537  0.028737   
tpm[3,2]                   0.810986   0.204199    0.290537  0.028737   
intercept[1]               4.172460   0.385268    0.595338  0.433438   
intercept[2]               4.479960   0.080766    0.184801  0.167500   
intercept[3]               4.152420   0.373807    0.546026  0.25

chain 1:   0%|          | 0/2000 [00:00<?, ?it/s, (Warmup)]



chain 1:   0%|          | 1/2000 [00:01<51:58,  1.56s/it, (Warmup)]

chain 1:   5%|▌         | 100/2000 [58:02<18:26:03, 34.93s/it, (Warmup)]


chain 1:  10%|█         | 200/2000 [1:39:23<14:29:00, 28.97s/it, (Warmup)]





chain 1:  15%|█▌        | 300/2000 [2:16:19<12:12:37, 25.86s/it, (Warmup)]


chain 1:  20%|██        | 400/2000 [2:40:44<9:31:26, 21.43s/it, (Warmup)] 




chain 1:  25%|██▌       | 500/2000 [3:05:21<7:55:40, 19.03s/it, (Warmup)]




chain 1:  30%|███       | 600/2000 [3:23:11<6:17:57, 16.20s/it, (Warmup)]


chain 1:  35%|███▌      | 700/2000 [3:43:04<5:20:41, 14.80s/it, (Warmup)]







chain 1:  40%|████      | 800/2000 [4:02:49<4:37:14, 13.86s/it, (Warmup)]




chain 1:  45%|████▌     | 900/2000 [4:23:14<4:04:54, 13.36s/it, (Warmup)]


chain 1:  50%|█████     | 1000/2000 [4:41:12<3:29:20, 12.56s/it, (Warmup)]

chain 1:  50%|█████     | 1001/2000 [4:41:29<3:29:27, 12.58s/it, (Sampling)]




chain 1:  5


11:18:13 - cmdstanpy - INFO - CmdStan done processing.
11:18:13 - cmdstanpy - WARNING - Non-fatal error during sampling:
Exception: ordered_probit: Cut-points is not a valid ordered vector. The element at 2 is inf, but should be greater than the previous element, inf (in 'hmm.stan', line 172, column 8 to column 74)
	Exception: ordered_probit: Cut-points is not a valid ordered vector. The element at 2 is inf, but should be greater than the previous element, inf (in 'hmm.stan', line 172, column 8 to column 74)
	Exception: ordered_probit: Cut-points is not a valid ordered vector. The element at 2 is inf, but should be greater than the previous element, inf (in 'hmm.stan', line 172, column 8 to column 74)
	Exception: ordered_probit: Cut-points is not a valid ordered vector. The element at 2 is inf, but should be greater than the previous element, inf (in 'hmm.stan', line 172, column 8 to column 74)
	Exception: ordered_probit: Cut-points is not a valid ordered vector. The element at 2 is i

11:18:13 - cmdstanpy - WARNING - Some chains may have failed to converge.
	Chain 1 had 10 divergent transitions (1.0%)
	Chain 2 had 5 divergent transitions (0.5%)
	Chain 2 had 976 iterations at max treedepth (97.6%)
	Chain 3 had 5 divergent transitions (0.5%)
	Chain 4 had 18 divergent transitions (1.8%)
	Use the "diagnose()" method on the CmdStanMCMC object to see further information.



✓ Sampling complete!
✓ Model saved to ../models/hmm_4_cmdstan.pkl

------------------------------------------------------------
Diagnostics:
------------------------------------------------------------
Checking sampler transitions treedepth.
977 of 4000 (24.43%) transitions hit the maximum treedepth limit of 10, or 2^10 leapfrog steps.
Trajectories that are prematurely terminated due to this limit will result in slow exploration.
For optimal performance, increase this limit.

Checking sampler transitions for divergences.
38 of 4000 (0.95%) transitions ended with a divergence.
These divergent transitions indicate that HMC is not fully able to explore the posterior distribution.
Try increasing adapt delta closer to 1.
If this doesn't remove all divergences, try to reparameterize the model.

Checking E-BFMI - sampler transitions HMC potential energy.
The E-BFMI, 0.01, is below the nominal threshold of 0.30 which suggests that HMC may have trouble exploring the target distribution.
If pos

## Training Summary


In [7]:
# Summary
print("\n" + "=" * 60)
print("TRAINING SUMMARY")
print("=" * 60)
print(f"\nTotal models trained: {len(trained_models)}")
print(f"Models: {list(trained_models.keys())}")
print(f"\nSaved in: {fitted_model_folder}")

# List saved models
saved_models = sorted(fitted_model_folder.glob("hmm_*_cmdstan.pkl"))
print(f"\nSaved HMM model files ({len(saved_models)}):")
for model_file in saved_models:
    size_mb = model_file.stat().st_size / (1024 * 1024)
    print(f"  - {model_file.name} ({size_mb:.2f} MB)")


TRAINING SUMMARY

Total models trained: 3
Models: ['hmm_2', 'hmm_3', 'hmm_4']

Saved in: ../models

Saved HMM model files (3):
  - hmm_2_cmdstan.pkl (90.88 MB)
  - hmm_3_cmdstan.pkl (106.90 MB)
  - hmm_4_cmdstan.pkl (123.06 MB)
